<a href="https://colab.research.google.com/github/tamil51118/ADP/blob/main/lceandlcd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random

# Define a Node in the network
class Node:
    def __init__(self, name):  # ✅ Fixed constructor
        self.name = name
        self.cache = set()  # simple cache with set of content names

    def has_content(self, content):
        return content in self.cache

    def cache_content(self, content):
        self.cache.add(content)

# ICN Network Simulation
class ICNNetwork:
    def __init__(self, num_nodes, strategy='LCE'):  # ✅ Fixed constructor
        self.nodes = [Node(f"Node{i}") for i in range(num_nodes)]
        self.producer = self.nodes[-1]  # Producer is the last node
        self.strategy = strategy
        self.cache_hits = 0
        self.cache_misses = 0

    def request_content(self, consumer_index, content):
        path = self.nodes[consumer_index:]  # from consumer to producer
        print(f"\nRequesting '{content}' from {self.nodes[consumer_index].name}")

        # Check for content along the path
        for i, node in enumerate(path):
            if node.has_content(content):
                print(f"✅ Cache HIT at {node.name}")
                self.cache_hits += 1

                if self.strategy == 'LCD':
                    if i > 0:  # leave copy at previous node
                        path[i-1].cache_content(content)
                        print(f"📥 Cached '{content}' at {path[i-1].name} (LCD)")
                return  # content served from cache

        # If not found, it's a cache miss
        print(f"❌ Cache MISS, content served by Producer {self.producer.name}")
        self.cache_misses += 1

        if self.strategy == 'LCE':
            # Cache the content at all nodes in the path
            for node in path:
                node.cache_content(content)
                print(f"📥 Cached '{content}' at {node.name} (LCE)")
        elif self.strategy == 'LCD':
            if len(path) > 1:
                path[-2].cache_content(content)
                print(f"📥 Cached '{content}' at {path[-2].name} (LCD)")

    def show_cache_status(self):
        print("\n📦 Current Cache Status:")
        for node in self.nodes:
            print(f"{node.name}: {node.cache}")

    def stats(self):
        print(f"\n📊 Cache Hits: {self.cache_hits}, Cache Misses: {self.cache_misses}")

# ------------------------
# Example usage

# Create a network of 5 nodes (Node0 to Node4), Node4 is the producer
print("=== LCE Strategy ===")
network_lce = ICNNetwork(num_nodes=5, strategy='LCE')
requests = ['A', 'B', 'A', 'C', 'B']

for content in requests:
    consumer_node = random.randint(0, 2)  # Node0, Node1 or Node2 make the request
    network_lce.request_content(consumer_node, content)

network_lce.show_cache_status()
network_lce.stats()

print("\n\n=== LCD Strategy ===")
network_lcd = ICNNetwork(num_nodes=5, strategy='LCD')

for content in requests:
    consumer_node = random.randint(0, 2)
    network_lcd.request_content(consumer_node, content)

network_lcd.show_cache_status()
network_lcd.stats()


=== LCE Strategy ===

Requesting 'A' from Node0
❌ Cache MISS, content served by Producer Node4
📥 Cached 'A' at Node0 (LCE)
📥 Cached 'A' at Node1 (LCE)
📥 Cached 'A' at Node2 (LCE)
📥 Cached 'A' at Node3 (LCE)
📥 Cached 'A' at Node4 (LCE)

Requesting 'B' from Node0
❌ Cache MISS, content served by Producer Node4
📥 Cached 'B' at Node0 (LCE)
📥 Cached 'B' at Node1 (LCE)
📥 Cached 'B' at Node2 (LCE)
📥 Cached 'B' at Node3 (LCE)
📥 Cached 'B' at Node4 (LCE)

Requesting 'A' from Node1
✅ Cache HIT at Node1

Requesting 'C' from Node1
❌ Cache MISS, content served by Producer Node4
📥 Cached 'C' at Node1 (LCE)
📥 Cached 'C' at Node2 (LCE)
📥 Cached 'C' at Node3 (LCE)
📥 Cached 'C' at Node4 (LCE)

Requesting 'B' from Node0
✅ Cache HIT at Node0

📦 Current Cache Status:
Node0: {'B', 'A'}
Node1: {'C', 'B', 'A'}
Node2: {'C', 'B', 'A'}
Node3: {'C', 'B', 'A'}
Node4: {'C', 'B', 'A'}

📊 Cache Hits: 2, Cache Misses: 3


=== LCD Strategy ===

Requesting 'A' from Node1
❌ Cache MISS, content served by Producer Node4
📥 C